# Parse Force : Deberta




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("=" * 60)
print(" Google Drive Mounted!")
print("=" * 60)
print("  All outputs → /content/drive/MyDrive/NLP_Project/")
print("  Protected from session disconnections.")
print("=" * 60)


## STEP 1: Configure Project Paths

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT       = "/content/drive/MyDrive/NLP_Project"
PROCESSED_DATA_DIR = f"{PROJECT_ROOT}/processed_data"
HF_DATASET_PATH    = f"{PROCESSED_DATA_DIR}/hf_dataset"   # Written by Member 1

MEMBER2_ROOT       = f"{PROJECT_ROOT}/models/member2"
DEBERTA_CKPT_DIR   = f"{MEMBER2_ROOT}/checkpoints/deberta"
DEBERTA_FINAL_DIR  = f"{MEMBER2_ROOT}/final_model/deberta"
SPANBERT_CKPT_DIR  = f"{MEMBER2_ROOT}/checkpoints/spanbert"
SPANBERT_FINAL_DIR = f"{MEMBER2_ROOT}/final_model/spanbert"
LOGITS_OUTPUT_PATH = f"{MEMBER2_ROOT}/deberta_logits_val.csv"  # → Member 3

for d in [DEBERTA_CKPT_DIR, DEBERTA_FINAL_DIR,
          SPANBERT_CKPT_DIR, SPANBERT_FINAL_DIR]:
    os.makedirs(d, exist_ok=True)

print("=" * 60)
print("MEMBER 2 — PROJECT STRUCTURE")
print("=" * 60)
print(f"  Input HF dataset : {HF_DATASET_PATH}")
print(f"  DeBERTa ckpts    : {DEBERTA_CKPT_DIR}")
print(f"  DeBERTa final    : {DEBERTA_FINAL_DIR}")
print(f"  SpanBERT ckpts   : {SPANBERT_CKPT_DIR}")
print(f"  SpanBERT final   : {SPANBERT_FINAL_DIR}")
print(f"  Logits CSV out   : {LOGITS_OUTPUT_PATH}")
print("=" * 60)


In [ ]:
import torch

print("=" * 60)
print("GPU CHECK")
print("=" * 60)
if torch.cuda.is_available():
    name   = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU  : {name}")
    print(f"   VRAM : {mem_gb:.2f} GB")
    if mem_gb < 14:
        print("  Less than 14 GB VRAM — consider reducing MAX_LENGTH to 192")
    else:
        print(" VRAM looks good for MAX_LENGTH=256!")
else:
    print(" No GPU detected — training will be extremely slow.")
    print("   Runtime → Change runtime type → GPU (T4)")
print("=" * 60)


## STEP 3: Install Dependencies


In [ ]:
# Pin versions identical to Member 1 + protobuf fix for DeBERTa-v3
!pip install -q \
    transformers==4.46.1 \
    accelerate==1.1.1 \
    datasets==3.1.0 \
    peft==0.13.2 \
    sentencepiece \
    "protobuf==3.20.*" \
    scikit-learn

# Verify the DeBERTa tokeniser import that was previously failing
try:
    from transformers import DebertaV2Tokenizer
    print(" DebertaV2Tokenizer import OK (protobuf fix confirmed working)")
except ImportError as e:
    print(f" Import still failing: {e}")
    print("   Try: !pip install protobuf==3.20.3 --force-reinstall")


## STEP 4: Shared Configuration

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
import json
from pathlib import Path

# ── Emotion registry — must match Member 1 exactly ───────────────────────
EMOTION_LIST = ["neutral", "anger", "disgust", "fear", "joy", "sadness", "surprise"]
EMOTION2ID   = {e: i for i, e in enumerate(EMOTION_LIST)}
ID2EMOTION   = {i: e for i, e in enumerate(EMOTION_LIST)}
NUM_LABELS   = len(EMOTION_LIST)   # 7

# ── Hyper-parameters ─────────────────────────────────────────────────────
class Config:
    MAX_LENGTH                  = 256
    BATCH_SIZE                  = 4
    GRADIENT_ACCUMULATION_STEPS = 8
    RANDOM_SEED                 = 42
    LEARNING_RATE_DEBERTA       = 2e-5
    LEARNING_RATE_SPANBERT      = 3e-5
    NUM_EPOCHS_DEBERTA          = 10
    NUM_EPOCHS_SPANBERT         = 5
    WARMUP_STEPS                = 500
    WEIGHT_DECAY                = 0.01
    SAVE_STEPS                  = 500
    EVAL_STEPS                  = 500
    LOGGING_STEPS               = 100
    SAVE_TOTAL_LIMIT            = 3
    DOC_STRIDE                  = 64
    INFERENCE_BATCH_SIZE        = 16

def set_seed(seed: int = Config.RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print(" Configuration loaded.")
print(f"   EMOTION_LIST = {EMOTION_LIST}")
print(f"   MAX_LENGTH   = {Config.MAX_LENGTH}")
print(f"   BATCH_SIZE   = {Config.BATCH_SIZE}")
print(f"   GRAD_ACCUM   = {Config.GRADIENT_ACCUMULATION_STEPS}")
print(f"   SAVE_TOTAL   = {Config.SAVE_TOTAL_LIMIT}  ← (was 2, raised to 3)")


In [ ]:
print(HF_DATASET_PATH)
!ls {HF_DATASET_PATH}

###The Bridge Fixer

In [ ]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict

def bridge_fix_flatten_spans(dataset_dict):
    """
    Extracts cause spans from the raw JSON if they are missing in the HF dataset.
    """
    # Check if we already have data to avoid unnecessary processing
    try:
        sample_span = dataset_dict["train"][0].get("cause_span", "")
        if sample_span and str(sample_span).strip() != "":
            return dataset_dict
    except:
        pass

    print(" No cause spans detected. Applying Bridge Fix from raw JSON...")

    # Path to your raw JSON file on Drive
    RAW_JSON_PATH = f"{PROJECT_ROOT}/data/Subtask_1_train.json"

    with open(RAW_JSON_PATH, 'r') as f:
        raw_data = json.load(f)

    rows = []
    for conv in raw_data:
        # Create a text lookup for this conversation
        u_map = {u['utterance_ID']: u['text'] for u in conv['conversation']}

        for pair in conv['emotion-cause_pairs']:
            # Extract target ID and emotion (e.g., "3_surprise" -> 3, "surprise")
            target_id = int(pair[0].split('_')[0])
            emotion = pair[0].split('_')[1]

            # Extract cause text and strip prefix (e.g., "1_Text" -> "Text")
            cause_text = pair[1].split('_', 1)[1] if '_' in pair[1] else pair[1]

            rows.append({
                "text": u_map[target_id],
                "emotion": emotion,
                "cause_span": cause_text,
                "has_cause": True,
                "emotion_label": EMOTION2ID.get(emotion, 0)
            })

    df = pd.DataFrame(rows)
    # Create 90/10 split
    train_df = df.sample(frac=0.9, random_state=42)
    val_df = df.drop(train_df.index)

    return DatasetDict({
        "train": Dataset.from_pandas(train_df),
        "validation": Dataset.from_pandas(val_df)
    })

In [ ]:
from datasets import load_from_disk, Features, Value

# Helper function to ensure cause_span is a string
# def bridge_fix_flatten_spans(dataset):
#     def _flatten_cause_span(example):
#         cause = example.get("cause_span")
#         if cause is None:
#             example["cause_span"] = ""
#         elif isinstance(cause, list):
#             example["cause_span"] = " ".join(str(item) for item in cause)
#         elif not isinstance(cause, str):
#             example["cause_span"] = str(cause)
#         return example
#     return dataset.map(_flatten_cause_span)

print(f"Loading dataset from:\n  {HF_DATASET_PATH}\n")
raw_dataset = load_from_disk(HF_DATASET_PATH)

# 1. Apply the Bridge Fix if spans are missing (from previous step)
raw_dataset = bridge_fix_flatten_spans(raw_dataset)

# 2. Normalise the values to Python booleans
def normalise_has_cause(batch):
    fixed = []
    for h in batch["has_cause"]:
        if h is None:
            fixed.append(False)
        elif isinstance(h, str):
            fixed.append(h.strip().lower() == "true")
        else:
            fixed.append(bool(h))
    return {"has_cause": fixed}

raw_dataset = raw_dataset.map(normalise_has_cause, batched=True, desc="Normalising values")

# ── THE CRITICAL FIX: Explicitly cast the Schema ──
# This tells Hugging Face: "I don't care what you thought this was, it is now a Boolean."
new_features = raw_dataset["train"].features.copy()
new_features["has_cause"] = Value("bool")
raw_dataset = raw_dataset.cast(new_features)

# 3. Final Verification
sample_val = raw_dataset["train"][0]["has_cause"]
sample_type = type(sample_val)

print("\n" + "=" * 60)
print(f" FINAL CHECK: value={sample_val}, type={sample_type.__name__}")
print("=" * 60)

if sample_type == bool:
    print(" SUCCESS: The schema is now correctly recognized as Boolean.")
else:
    print(" FAILED: Still showing as", sample_type.__name__)

# Dataset summary
for split, ds in raw_dataset.items():
    n_with_cause = sum(1 for x in ds["has_cause"] if x is True)
    print(f"  {split:12s}: {len(ds):>6,} total  |  {n_with_cause:>5,} with cause span")

## STEP 5: Load & Validate Dataset


In [ ]:
from datasets import load_from_disk, Features, Value

print(f"Loading dataset from:\n  {HF_DATASET_PATH}\n")
raw_dataset = load_from_disk(HF_DATASET_PATH)

# 1. Apply the Bridge Fix (this populates the 8,428 spans you saw earlier)
raw_dataset = bridge_fix_flatten_spans(raw_dataset)

# 2. Normalise values and FORCE the Schema to Boolean
def normalise_has_cause(batch):
    return {"has_cause": [bool(h) for h in batch["has_cause"]]}

raw_dataset = raw_dataset.map(normalise_has_cause, batched=True, desc="Fixing Boolean values")

# THE KEY STEP: This removes the "Still not bool" warning permanently
new_features = raw_dataset["train"].features.copy()
new_features["has_cause"] = Value("bool")
raw_dataset = raw_dataset.cast(new_features)

# 3. Summary Verification
print("\n" + "=" * 60)
print("DATASET OVERVIEW (POST-FIX)")
print("=" * 60)
for split, ds in raw_dataset.items():
    n_with_cause = sum(1 for x in ds["has_cause"] if x is True)
    sample_type = type(ds[0]["has_cause"])
    print(f"  {split:12s}: {len(ds):>6,} total | {n_with_cause:>5,} with cause span | Type: {sample_type.__name__}")

## STEP 6: Level 0 — DeBERTa-v3 Syntactic Expert



In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
)
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

DEBERTA_MODEL_NAME = "microsoft/deberta-v3-base"

print(f"Loading DeBERTa tokeniser: {DEBERTA_MODEL_NAME}")
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_NAME)
print(" DeBERTa tokeniser loaded.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 6.1  Tokenisation
# Context + utterance joined with [SEP] — DeBERTa's native two-segment input
# ─────────────────────────────────────────────────────────────────────────
def tokenize_for_deberta(examples):
    """
    Combines 3-utterance backward context with the target utterance.
    Uses DeBERTa's native [SEP] separator (not RoBERTa's </s>).
    """
    inputs = []
    contexts = examples.get("context", [""] * len(examples["text"]))
    for ctx, txt in zip(contexts, examples["text"]):
        ctx = ctx or ""
        inputs.append(f"{ctx} [SEP] {txt}" if ctx else txt)

    tokenized = deberta_tokenizer(
        inputs,
        padding="max_length",
        truncation=True,
        max_length=Config.MAX_LENGTH,
        return_tensors=None,
    )
    tokenized["labels"] = examples["emotion_label"]
    return tokenized

print("Tokenising for DeBERTa...")
deberta_dataset = raw_dataset.map(
    tokenize_for_deberta,
    batched=True,
    remove_columns=[c for c in raw_dataset["train"].column_names
                    if c != "emotion_label"],
    desc="DeBERTa tokenisation",
)
deberta_dataset.set_format("torch")
print(f" DeBERTa dataset ready — Train: {len(deberta_dataset['train']):,} | Val: {len(deberta_dataset['validation']):,}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────
train_labels_list = raw_dataset["train"]["emotion_label"]

# Get the unique labels actually present in the training data
unique_labels_in_y = np.unique(train_labels_list)

# Compute weights only for the labels present in the training data
# This avoids the ValueError if some labels from 0-6 are missing from train_labels_list
present_class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=unique_labels_in_y,
    y=train_labels_list,
)

# Create a full array of weights for all NUM_LABELS (0-6)
# Initialize with a default value, e.g., 1.0 (no explicit weighting if missing)
class_weights_full = np.ones(NUM_LABELS, dtype=np.float32)

# Map the computed weights back to the full set of labels
for i, label in enumerate(unique_labels_in_y):
    class_weights_full[label] = present_class_weights_np[i]

class_weights_tensor = torch.tensor(class_weights_full, dtype=torch.float32)

print("=" * 60)
print("CLASS WEIGHTS (balanced)")
print("=" * 60)
for emotion_id, emotion_name in ID2EMOTION.items():
    w = class_weights_full[emotion_id] # Use the full array for printing
    bar = "█" * int(w * 10)
    print(f"  {emotion_name:10s}: {w:.4f}  {bar}")
print()
print("These upweight rare classes (Disgust, Fear) and downweight Joy/Neutral.")
print("Missing classes (if any) are assigned a weight of 1.0.")


In [ ]:

# Overrides compute_loss() to apply per-class weights to CrossEntropyLoss.
# ─────────────────────────────────────────────────────────────────────────
class WeightedTrainer(Trainer):
    """Trainer subclass that applies class-weighted CrossEntropyLoss."""

    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Store weights on the correct device; will be moved in compute_loss
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        # Move weights to same device as logits
        weights = self.class_weights.to(logits.device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


print(" WeightedTrainer defined.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 6.4  Metrics
# ─────────────────────────────────────────────────────────────────────────
def compute_emotion_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)

    metrics = {
        "f1_weighted":        float(f1),
        "precision_weighted": float(precision),
        "recall_weighted":    float(recall),
    }
    for i, emotion in enumerate(EMOTION_LIST):
        if i < len(per_class_f1):
            metrics[f"f1_{emotion}"] = float(per_class_f1[i])
    return metrics

print(" compute_emotion_metrics defined.")


In [ ]:
# ── REPLACEMENT: Load checkpoint-2500 as final DeBERTa model ─────────────
# Skips the remaining 130 training steps entirely.
# checkpoint-2500 is at 95% and sufficient for logit generation.

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
)
import os

# ── Verify checkpoint exists on Drive ────────────────────────────────────
ckpt_path = os.path.join(DEBERTA_CKPT_DIR, "checkpoint-2500")
final_path = DEBERTA_FINAL_DIR

if not os.path.exists(ckpt_path):
    # Fall back to whatever checkpoint is latest
    from pathlib import Path
    available = sorted(
        [d for d in Path(DEBERTA_CKPT_DIR).iterdir()
         if d.name.startswith("checkpoint-")],
        key=lambda x: int(x.name.split("-")[1])
    )
    if available:
        ckpt_path = str(available[-1])
        print(f"  checkpoint-2500 not found, using: {ckpt_path}")
    else:
        raise FileNotFoundError(
            f"No checkpoints found in {DEBERTA_CKPT_DIR}\n"
            "Make sure Drive is mounted and the path is correct."
        )

print(f"Loading DeBERTa from: {ckpt_path}")

# ── Load model from checkpoint, tokenizer from HuggingFace ───────────────
# Checkpoints only contain model weights, not tokenizer files.
# Tokenizer hasn't changed from fine-tuning so we reload from base model.
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    ckpt_path,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True,
)
deberta_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

# Save both to final_model dir so all downstream cells use standard path
deberta_model.save_pretrained(final_path)
deberta_tokenizer.save_pretrained(final_path)

print(f"✅ Model loaded from     : {ckpt_path}")
print(f"✅ Tokenizer loaded from : microsoft/deberta-v3-base")
print(f"✅ Both saved to         : {final_path}")

# ── Rebuild TrainingArguments (needed for Trainer.evaluate/predict) ───────
deberta_training_args = TrainingArguments(
    output_dir=DEBERTA_CKPT_DIR,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    fp16=False,
    report_to="none",
    seed=Config.RANDOM_SEED,
)

# ── Rebuild Trainer (no training — just for evaluate() and predict()) ─────
deberta_trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=deberta_model,
    args=deberta_training_args,
    train_dataset=deberta_dataset["train"],
    eval_dataset=deberta_dataset["validation"],
    compute_metrics=compute_emotion_metrics,
)

print("\n" + "=" * 60)
print(" DeBERTa ready — skipped final 130 steps")
print("=" * 60)
print(f"  Source    : {ckpt_path}")
print(f"  Final dir : {final_path}")
print("  Next: run deberta_eval → deberta_logits → vram_cleanup")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 6.5  Model + TrainingArguments

# fp16=False kept — DeBERTa-v3 can diverge silently with fp16
# ─────────────────────────────────────────────────────────────────────────
print(f"Loading model: {DEBERTA_MODEL_NAME}")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True,
)
total_params = sum(p.numel() for p in deberta_model.parameters()) / 1e6
print(f" DeBERTa loaded — {total_params:.1f}M parameters")

deberta_training_args = TrainingArguments(
    output_dir=DEBERTA_CKPT_DIR,
    num_train_epochs=Config.NUM_EPOCHS_DEBERTA,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=Config.LEARNING_RATE_DEBERTA,
    warmup_steps=Config.WARMUP_STEPS,
    weight_decay=Config.WEIGHT_DECAY,
    logging_dir=f"{DEBERTA_CKPT_DIR}/logs",
    logging_steps=Config.LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=Config.EVAL_STEPS,
    save_strategy="steps",
    save_steps=Config.SAVE_STEPS,
    save_total_limit=Config.SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    fp16=False,
    report_to="none",
    seed=Config.RANDOM_SEED,
)

deberta_trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=deberta_model,
    args=deberta_training_args,
    train_dataset=deberta_dataset["train"],
    eval_dataset=deberta_dataset["validation"],
    compute_metrics=compute_emotion_metrics,
)

print("\n" + "=" * 60)
print("DEBERTA TRAINING CONFIG")
print("=" * 60)
print(f"  Epochs         : {Config.NUM_EPOCHS_DEBERTA}")
print(f"  Effective batch: {Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS}")
print(f"  LR             : {Config.LEARNING_RATE_DEBERTA}")
print(f"  fp16           : False (intentional)")
print(f"  save_total_limit: {Config.SAVE_TOTAL_LIMIT}  ← (was 2)")
print(f"  Class weights  :  applied")
print("=" * 60)


In [ ]:
# RESUME: find the latest checkpoint on Drive
import os
from pathlib import Path

ckpt_dir = Path(DEBERTA_CKPT_DIR)
checkpoints = sorted(
    [d for d in ckpt_dir.iterdir() if d.name.startswith("checkpoint-")],
    key=lambda x: int(x.name.split("-")[1])
)

if checkpoints:
    latest_ckpt = str(checkpoints[-1])
    print(f" Resuming from: {latest_ckpt}")
else:
    latest_ckpt = None
    print("  No checkpoint found — will train from scratch")

In [ ]:
# Fix: Reset AcceleratorState before resuming
from accelerate.state import AcceleratorState, PartialState

# Wipe the broken singleton state left over from the interrupted session
AcceleratorState._reset_state(reset_partial_state=True)

# Reinitialise cleanly
_ = PartialState()

print(" AcceleratorState reset and reinitialised.")
print(f"   Distributed type: {PartialState().distributed_type}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 6.6  Train DeBERTa  ⏱ ~2–3 h on T4
# ─────────────────────────────────────────────────────────────────────────
# import torch
# import numpy as np

# torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])

set_seed()

print(" Starting DeBERTa training...\n")
deberta_train_result = deberta_trainer.train(resume_from_checkpoint=latest_ckpt)
#deberta_train_result = deberta_trainer.train(resume_from_checkpoint=True)

deberta_trainer.save_model(DEBERTA_FINAL_DIR)
deberta_tokenizer.save_pretrained(DEBERTA_FINAL_DIR)

print("\n" + "=" * 60)
print(" DEBERTA TRAINING COMPLETE")
print("=" * 60)
print(f"  Training loss : {deberta_train_result.training_loss:.4f}")
print(f"  Saved to      : {DEBERTA_FINAL_DIR}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 6.7  Evaluate & classification report
# ─────────────────────────────────────────────────────────────────────────
print("Evaluating DeBERTa on validation set...")
deberta_eval = deberta_trainer.evaluate(deberta_dataset["validation"])

print("\n" + "=" * 60)
print("DEBERTA EVALUATION RESULTS")
print("=" * 60)
for k, v in deberta_eval.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Detailed per-class report
preds_out   = deberta_trainer.predict(deberta_dataset["validation"])
pred_labels = np.argmax(preds_out.predictions, axis=-1)
true_labels = preds_out.label_ids

# Check which classes actually appear in validation
unique_true = sorted(set(true_labels.tolist()))
unique_pred = sorted(set(pred_labels.tolist()))
all_classes = sorted(set(unique_true) | set(unique_pred))
print(f"\n  Classes in val labels : {[EMOTION_LIST[i] for i in unique_true]}")
print(f"  Classes in val preds  : {[EMOTION_LIST[i] for i in unique_pred]}")
missing = [EMOTION_LIST[i] for i in range(NUM_LABELS) if i not in unique_true]
if missing:
    print(f"    Missing from val set: {missing} (zero examples — normal for rare classes)")

# FIX: pass labels= explicitly so sklearn knows to expect all 7 classes
# even if some have zero true examples in the validation split
report = classification_report(
    true_labels,
    pred_labels,
    labels=list(range(NUM_LABELS)),      # force all 7 class indices
    target_names=EMOTION_LIST,           # map indices to emotion names
    digits=4,
    zero_division=0,                     # show 0.0 instead of warning for missing classes
)
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(report)

report_path = Path(DEBERTA_FINAL_DIR) / "classification_report_val.txt"
with open(report_path, "w") as f:
    f.write(report)

eval_path = Path(DEBERTA_FINAL_DIR) / "eval_results_val.json"
with open(eval_path, "w") as f:
    json.dump({k: v for k, v in deberta_eval.items()}, f, indent=2)

print(f" Report  → {report_path}")
print(f" Metrics → {eval_path}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# 6.8  Export logits CSV for Member 3  (PRIMARY DELIVERABLE)
# ─────────────────────────────────────────────────────────────────────────
print("Generating deberta_logits_val.csv for Member 3...")

preds_out   = deberta_trainer.predict(deberta_dataset["validation"])
logits      = preds_out.predictions          # (N, 7)
true_labels = preds_out.label_ids
pred_labels = np.argmax(logits, axis=-1)
probs       = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
confidences = np.max(probs, axis=-1)

logits_df = pd.DataFrame({
    "true_label":   true_labels,
    "pred_label":   pred_labels,
    "true_emotion": [ID2EMOTION[l] for l in true_labels],
    "pred_emotion": [ID2EMOTION[l] for l in pred_labels],
    "confidence":   confidences,
})
for i, emotion in enumerate(EMOTION_LIST):
    logits_df[f"logit_{emotion}"] = logits[:, i]
for i, emotion in enumerate(EMOTION_LIST):
    logits_df[f"prob_{emotion}"]  = probs[:, i]

logits_df.to_csv(LOGITS_OUTPUT_PATH, index=False)

print("\n" + "=" * 60)
print(" LOGITS CSV EXPORTED")
print("=" * 60)
print(f"  Rows    : {len(logits_df):,}")
print(f"  Columns : {list(logits_df.columns)}")
print(f"  Path    : {LOGITS_OUTPUT_PATH}")
print(f"  Accuracy: {(pred_labels == true_labels).mean():.4f}")
print(f"  Avg conf: {confidences.mean():.4f}")
print("\n   Share this path with Member 3!")


## VRAM Cleanup Before SpanBERT


In [ ]:
import gc


del deberta_model
del deberta_trainer

gc.collect()
torch.cuda.empty_cache()

# Verify memory freed
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0)  / 1e9
    print("=" * 60)
    print(" VRAM FREED  ← FIX 3")
    print("=" * 60)
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")
    print("  Ready to load SpanBERT safely.")
    print("=" * 60)
else:
    print(" Cleanup done (no GPU to clear).")
